# 09 — OE Superpixel Inversion (SLIC + PCA/kNN)

Demonstrates the SLIC superpixel inversion pipeline from `superpixel_oe.py`:

1. **Segment** the image into spectrally coherent superpixels (SLIC)
2. **Invert** the mean spectrum of each superpixel with OE
3. **Back-interpolate** to full pixel resolution using PCA + kNN IDW

Same Sentinel-2 Wadden Sea scene and configuration as NB07b.

| Step | Key parameter | Paper default |
|---|---|---|
| SLIC `n_segments` | target superpixel count | ~1% of pixels |
| SLIC `compactness` | shape vs spectral fidelity | 0.1 (low → follow edges) |
| SLIC `sigma` | pre-smoothing | 2.0 |
| PCA `n_components` | spectral embedding dims | 6 |
| kNN `k` | neighbours for IDW | 4 |

Reference: Adams et al. (2021), *Remote Sensing of Environment*.

In [ ]:
import os
import time
import numpy as np
import xarray as xr
import lmfit
import matplotlib.pyplot as plt
import jax.numpy as jnp
import cmocean.cm as cm
from skimage.segmentation import mark_boundaries

from bio_optics.inversion import oe_engine, dask_oe_engine, superpixel_oe
from bio_optics.reflectance import albert_mobley_jax

## Load data

Same Sentinel-2 Wadden Sea scene as NB07b.

In [ ]:
dataset = xr.open_dataset(os.path.join(os.getcwd(), 'example_data/S2L2A_example.nc'))

wavelengths = np.array([490, 560, 665, 705, 740, 783, 842, 865])
band_vars   = list(dataset.data_vars)[1:]

ref_band          = dataset.isel(time=1)[band_vars[0]]
spatial_dim_names = list(ref_band.dims)
spatial_coords    = {d: dataset[d].values for d in spatial_dim_names if d in dataset.coords}

Rrs_image = np.stack(
    [dataset.isel(time=1)[v].values for v in band_vars], axis=-1
) / np.pi

n_rows, n_cols, n_obs = Rrs_image.shape
n_pixels = n_rows * n_cols
print(f'Rrs_image: {Rrs_image.shape}  ({n_pixels} pixels, {n_obs} bands)')

## Configuration (identical to NB07b)

In [ ]:
_LUT_NAMES   = ['const10%', 'sand', 'coral', 'CCA', 'macrophyte', 'seagrass']
bottom_types = [1, 5]   # sand + seagrass

fit_config = {
    'C_0':     dict(vary=False, value=0.5,  sigma_a=1.0, log=True),
    'C_Y':     dict(vary=False, value=0.1,  sigma_a=1.0, log=True),
    'C_Mie':   dict(vary=False, value=0.1,  sigma_a=1.0, log=True),
    'zB':      dict(vary=True,  value=0.5,  sigma_a=0.7, log=True),
    'f_mix_0': dict(vary=True,  value=0.0,  sigma_a=2.0, log=False),
}

noise     = 0.0031
n_iter    = 15
tile_size = 4096

PARAM_META = {
    'zB':      ('Depth z_B',          'm',     cm.deep),
    'f_mix_0': ('Mix logit (sand)',    'logit', 'RdYlGn'),
}

# --- build lmfit Parameters --------------------------------------------------
params = lmfit.Parameters()
params.add('theta_sun',  value=np.radians(30), vary=False)
params.add('theta_view', value=np.radians(0),  vary=False)
params.add('n1',         value=1.0,            vary=False)
params.add('n2',         value=1.33,           vary=False)
params.add('kappa_0',    value=1.0546,         vary=False)
params.add('C_0',   value=fit_config['C_0']['value'],   vary=fit_config['C_0']['vary'])
params.add('C_Y',   value=fit_config['C_Y']['value'],   vary=fit_config['C_Y']['vary'])
params.add('C_Mie', value=fit_config['C_Mie']['value'], vary=fit_config['C_Mie']['vary'])
for c in range(6):
    params.add(f'C_{c+1}', value=0.0, vary=False)
params.add('C_X', value=0.0, vary=False)
params.add('S',                   value=0.014,  vary=False)
params.add('S_NAP',               value=0.011,  vary=False)
params.add('lambda_0',            value=440.0,  vary=False)
params.add('K',                   value=0.0,    vary=False)
params.add('T_W',                 value=18.0,   vary=False)
params.add('T_W_0',               value=20.0,   vary=False)
params.add('a_NAP_spec_lambda_0', value=0.041,  vary=False)
params.add('bb_phy_spec',         value=0.0010, vary=False)
params.add('bb_Mie_spec',         value=0.0042, vary=False)
params.add('bb_X_spec',           value=0.0086, vary=False)
params.add('lambda_S',            value=500.0,  vary=False)
params.add('n',                   value=-1.0,   vary=False)
for i in range(6):
    params.add(f'f_{i}', value=1.0 if i == 0 else 0.0, vary=False)
    params.add(f'B_{i}', value=1/np.pi, vary=False)
params.add('f_mix_0', value=fit_config['f_mix_0']['value'], vary=True)
params.add('zB',      value=fit_config['zB']['value'],      vary=True)

sigma_a    = {n: c['sigma_a'] for n, c in fit_config.items() if c['vary']}
log_params = [n for n, c in fit_config.items() if c['vary'] and c['log']]

# --- precompute + remap bottom LUT ------------------------------------------
pre_base     = albert_mobley_jax.precompute(wavelengths)
R_b_full     = np.array(pre_base['R_b_i'])
R_b_selected = np.zeros_like(R_b_full)
for dst, src in enumerate(bottom_types):
    R_b_selected[:, dst] = R_b_full[:, src]
pre = {**pre_base, 'R_b_i': jnp.array(R_b_selected)}

all_names = list(params.keys())
f_vec     = albert_mobley_jax.make_forward_vec(all_names, pre)
setup     = oe_engine.build_inversion(params, f_vec, sigma_a, log_params=log_params)

print('Free parameters:', setup.fit_names)
print('log_params:',      log_params)

## Step 1 — SLIC segmentation

`compactness=0.1` allows superpixel shapes to follow natural spectral edges  
(water eddies, depth gradients) rather than enforcing compact blobs.  
`sigma=2.0` reduces the impact of per-pixel spectral noise on cluster assignment.

In [ ]:
# SLIC parameters (paper defaults)
n_segments  = 1200   # ~1% of 119 883 pixels
compactness = 0.1
slic_sigma  = 2.0

t0     = time.perf_counter()
labels = superpixel_oe.segment_image(
    Rrs_image, n_segments=n_segments,
    compactness=compactness, sigma=slic_sigma,
)
t_seg  = time.perf_counter() - t0

n_segs_actual = len(np.unique(labels))
print(f'Requested {n_segments} segments → got {n_segs_actual}  ({t_seg:.2f} s)')
print(f'Mean pixels per segment: {n_pixels / n_segs_actual:.0f}')

In [ ]:
# False-colour RGB for overlay
rgb_bands = [2, 1, 0]   # B04, B03, B02 → R, G, B
rgb = Rrs_image[..., rgb_bands]
rgb = (rgb - np.nanpercentile(rgb, 2)) / (np.nanpercentile(rgb, 98) - np.nanpercentile(rgb, 2))
rgb = np.clip(rgb, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(rgb, origin='upper')
axes[0].set_title('Rrs false colour (B04/B03/B02)')
axes[0].axis('off')

# mark_boundaries draws segment outlines on the image
img_boundaries = mark_boundaries(rgb, labels, color=(1, 1, 0), mode='thick')
axes[1].imshow(img_boundaries, origin='upper')
axes[1].set_title(f'SLIC segments (n={n_segs_actual}, compactness={compactness}, σ={slic_sigma})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Step 2 — Aggregate superpixel mean spectra

In [ ]:
sp_spectra, sp_counts = superpixel_oe.aggregate_superpixels(Rrs_image, labels)

print(f'sp_spectra shape: {sp_spectra.shape}')
print(f'Pixel counts — min: {sp_counts.min()}  median: {np.median(sp_counts):.0f}  max: {sp_counts.max()}')

fig, ax = plt.subplots(figsize=(8, 4))
cvals = np.linspace(0, 1, n_segs_actual)
for i in range(n_segs_actual):
    ax.plot(wavelengths, sp_spectra[i], color=plt.cm.viridis(cvals[i]), alpha=0.3, lw=0.5)
ax.plot(wavelengths, sp_spectra.mean(axis=0), 'k-', lw=2, label='scene mean')
ax.set_xlabel('Wavelength [nm]')
ax.set_ylabel('Rrs [sr⁻¹]')
ax.set_title(f'Superpixel mean spectra (n={n_segs_actual})')
ax.legend()
plt.tight_layout()
plt.show()

## Step 3 — OE inversion of superpixel mean spectra

Each superpixel is inverted independently using the same `setup`.  
The noise passed here is the **original pixel noise** — the chi2 values will be  
smaller than for per-pixel inversion because mean spectra are smoother.  
The calibrated chi2 = chi2_raw × N_pixels accounts for this.

In [ ]:
t0 = time.perf_counter()
sp_results = superpixel_oe.invert_superpixels(
    sp_spectra, sp_counts, setup, noise,
    n_iter=n_iter, tile_size=tile_size,
)
t_sp = time.perf_counter() - t0

print(f'Superpixel inversion: {n_segs_actual} segments in {t_sp:.2f} s')
print(f'Speedup vs full image (~{n_pixels} px): {n_pixels / n_segs_actual:.0f}× fewer inversions')

In [ ]:
chi2_raw = sp_results['chi2']
chi2_cal = chi2_raw * sp_counts   # calibrated: chi2 × N

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

kw = dict(bins=40, density=True, alpha=0.7)
axes[0].hist(chi2_raw, **kw, color='steelblue', label=f'median={np.median(chi2_raw):.3f}')
axes[0].axvline(1.0, color='r', ls='--', lw=1.5, label='ideal χ²=1')
axes[0].set_xlabel('χ² (raw, pixel noise)')
axes[0].set_ylabel('Density')
axes[0].set_title('Superpixel chi2 (raw)')
axes[0].legend()

axes[1].hist(chi2_cal, range=(0, 5), **kw, color='tomato', label=f'median={np.median(chi2_cal):.3f}')
axes[1].axvline(1.0, color='r', ls='--', lw=1.5, label='ideal χ²=1')
axes[1].set_xlabel('χ² calibrated (= χ²_raw × N)')
axes[1].set_ylabel('Density')
axes[1].set_title('Superpixel chi2 (calibrated for mean-spectrum noise)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Median chi2 raw:        {np.median(chi2_raw):.4f}')
print(f'Median chi2 calibrated: {np.median(chi2_cal):.4f}  (should be ≈ 1 if noise is well-calibrated)')

## Step 4 — Back-interpolation: PCA + kNN IDW

For each pixel, find the k=4 nearest superpixels in 6-component PCA space  
(fitted on brightness-normalised mean spectra), then compute an inverse-distance  
weighted average of their retrieval results.

Uncertainty propagation:
- **Formal**: `σ²_pixel = Σ wᵢ² σ²_spᵢ` — posterior uncertainty propagated through IDW weights
- **Interpolation spread**: `+ Σ wᵢ (x_hat_spᵢ − x_hat_pixel)²` — spread among k neighbours

In [ ]:
k            = 4
n_components = 6

t0 = time.perf_counter()
x_hat_flat, sigma_flat, A_diag_flat = superpixel_oe.backinterp_pca_knn(
    Rrs_image.reshape(n_pixels, n_obs),
    sp_spectra,
    sp_results['x_hat'],
    sp_results['sigma'],
    sp_results['A_diag'],
    k=k, n_components=n_components,
)
t_interp = time.perf_counter() - t0

n_fit   = x_hat_flat.shape[-1]
x_hat_sp  = x_hat_flat.reshape(n_rows, n_cols, n_fit)
sigma_sp  = sigma_flat.reshape(n_rows, n_cols, n_fit)
A_diag_sp = A_diag_flat.reshape(n_rows, n_cols, n_fit)

print(f'Back-interpolation: {t_interp:.2f} s')
print(f'x_hat shape: {x_hat_sp.shape}')

### PCA embedding — superpixels coloured by retrieval

In [ ]:
from sklearn.decomposition import PCA

sp_norm = sp_spectra / np.where(sp_spectra.sum(axis=-1, keepdims=True) == 0, 1.0,
                                sp_spectra.sum(axis=-1, keepdims=True))
pca    = PCA(n_components=min(n_components, n_segs_actual - 1, n_obs))
sp_pca = pca.fit_transform(sp_norm)

param_labels = {n: PARAM_META[n] for n in setup.fit_names if n in PARAM_META}
n_params = len(param_labels)

fig, axes = plt.subplots(1, n_params, figsize=(6 * n_params, 5))
axes = np.atleast_1d(axes)

for ax, (param, (label, unit, cmap)) in zip(axes, param_labels.items()):
    i     = setup.fit_names.index(param)
    color = sp_results['x_hat'][:, i]
    sc    = ax.scatter(sp_pca[:, 0], sp_pca[:, 1], c=color, cmap=cmap, s=20, alpha=0.8)
    plt.colorbar(sc, ax=ax, label=unit)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f'Superpixels coloured by {label}')

plt.tight_layout()
plt.show()

## Maps — superpixel vs per-pixel (NB07b reference)

Run the reference per-pixel inversion for direct comparison.

In [ ]:
t0 = time.perf_counter()
results_px = dask_oe_engine.invert_image(
    Rrs_image, setup, noise,
    n_iter=n_iter, tile_size=tile_size,
)
t_px = time.perf_counter() - t0
print(f'Per-pixel inversion: {t_px:.2f} s')

In [ ]:
for param, (label, unit, cmap) in param_labels.items():
    i    = setup.fit_names.index(param)
    px   = results_px['x_hat'][..., i]
    sp   = x_hat_sp[..., i]
    diff = px - sp

    vmin, vmax = np.nanpercentile(px, [2, 98])
    dabs = np.nanpercentile(np.abs(diff), 98)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    im0 = axes[0].imshow(px,   vmin=vmin, vmax=vmax, cmap=cmap, origin='upper')
    im1 = axes[1].imshow(sp,   vmin=vmin, vmax=vmax, cmap=cmap, origin='upper')
    im2 = axes[2].imshow(diff, vmin=-dabs, vmax=dabs, cmap='RdBu_r', origin='upper')

    for ax, im, title in zip(axes, [im0, im1, im2],
                              ['per-pixel OE', 'superpixel OE', 'difference (px − sp)']):
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_title(f'{label} [{unit}] — {title}')
        ax.axis('off')

    rmsd = np.sqrt(np.nanmean(diff**2))
    print(f'{param}: RMSD = {rmsd:.4g}  max|diff| = {np.nanmax(np.abs(diff)):.4g}')

    plt.tight_layout()
    plt.show()

## Uncertainty comparison — propagated σ

In [ ]:
for param, (label, unit, _) in param_labels.items():
    i      = setup.fit_names.index(param)
    sig_px = results_px['sigma'][..., i]
    sig_sp = sigma_sp[..., i]

    vmax = np.nanpercentile(sig_px, 98)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    im0 = axes[0].imshow(sig_px, vmin=0, vmax=vmax, cmap='Purples', origin='upper')
    im1 = axes[1].imshow(sig_sp, vmin=0, vmax=vmax, cmap='Purples', origin='upper')
    for ax, im, title in zip(axes, [im0, im1], ['per-pixel OE', 'superpixel propagated']):
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_title(f'σ({label}) [{unit}] — {title}')
        ax.axis('off')

    print(f'{param}: median σ  px={np.nanmedian(sig_px):.4f}  sp={np.nanmedian(sig_sp):.4f}')
    plt.tight_layout()
    plt.show()

## Timing summary

In [ ]:
t_total_sp = t_seg + t_sp + t_interp

print(f'\n{"Step":<30s} {"Time":>8s}')
print('-' * 40)
print(f'{"SLIC segmentation":<30s} {t_seg:>8.2f} s')
print(f'{"Superpixel OE inversion":<30s} {t_sp:>8.2f} s  ({n_segs_actual} segments)')
print(f'{"PCA + kNN back-interpolation":<30s} {t_interp:>8.2f} s')
print(f'{"Total superpixel pipeline":<30s} {t_total_sp:>8.2f} s')
print(f'{"Per-pixel OE (reference)":<30s} {t_px:>8.2f} s  ({n_pixels} pixels)')
print()
print(f'Speedup: {t_px / t_total_sp:.1f}×  ({n_pixels / n_segs_actual:.0f}× fewer OE inversions)')

## Quality diagnostics — within-segment variance

Within-segment variance of the back-interpolated x_hat measures how much  
the kNN back-interpolation introduces spatial variability inside each segment.  
Ideal: low within-segment variance (smooth within homogeneous segments),  
higher near spectral edges (where neighbouring segments have different values).

In [ ]:
labels_flat = labels.ravel()
seg_ids     = np.unique(labels_flat)

for param, (label, unit, cmap) in param_labels.items():
    i          = setup.fit_names.index(param)
    x_flat     = x_hat_sp[..., i].ravel()
    within_var = np.zeros(n_rows * n_cols)

    for sid in seg_ids:
        mask = labels_flat == sid
        within_var[mask] = x_flat[mask].var()

    within_std = np.sqrt(within_var).reshape(n_rows, n_cols)

    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(within_std, cmap='hot_r', origin='upper',
                   vmax=np.nanpercentile(within_std, 98))
    plt.colorbar(im, ax=ax, label=unit)
    ax.set_title(f'Within-segment std of {label} (low = homogeneous segments)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f'{param}: median within-segment std = {np.nanmedian(within_std):.4g} {unit}')